In [1]:
import enum
import os
from copy import deepcopy

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.model import train_one_epoch, validate
from internal.nn.test_time_augmentation import apply_tta
from internal.nn.weighted_random_sampler import make_weighted_sampler
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
def get_classifier_module(model: nn.Module):
    # Common names in timm models
    for name in ["classifier", "fc", "head"]:
        if hasattr(model, name):
            return getattr(model, name), name
    # Fallback: assume there is a single linear at the very end
    last_linear = None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Linear):
            last_linear = m
            break
    if last_linear is None:
        raise RuntimeError("Could not find classifier layer in model.")
    return last_linear, None

In [3]:
class PreTrainedArchitectures(enum.Enum):
    EFFICIENTNETV2_S = "tf_efficientnetv2_s.in21k"
    CONVNEXT_TINY = "convnext_tiny"
    EFFICIENTNET_B0 = "efficientnet_b0"
    EFFICIENTNET_B1 = "efficientnet_b1"

MODEL_TO_USE: PreTrainedArchitectures = PreTrainedArchitectures.EFFICIENTNET_B0

In [4]:
best_f1_per_fold: dict[int, int] = {}

In [5]:
def create_efficientnet_b0_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.2,        # Dropout
        # drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

def freeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_block_and_head(model: nn.Module):
    """
    For Stage 1: unfreeze last block + classifier head.
    """
    freeze_all(model)

    # last conv block
    if hasattr(model, "blocks"):
        for p in model.blocks[-1].parameters():
            p.requires_grad = True

    # classifier / head
    clf_module, _ = get_classifier_module(model)
    for p in clf_module.parameters():
        p.requires_grad = True

def unfreeze_last_two_blocks_and_head(model: nn.Module):
    """
    For Stage 2: unfreeze last 2 blocks + classifier head.
    """
    freeze_all(model)

    if hasattr(model, "blocks"):
        for blk in model.blocks[-2:]:
            for p in blk.parameters():
                p.requires_grad = True

    clf_module, _ = get_classifier_module(model)
    for p in clf_module.parameters():
        p.requires_grad = True


if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 or MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 5
    LR_STAGE1 = 1e-3
    LR_STAGE2 = 1e-4
    PREFIX = "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1"

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,  # Augmentations applied
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False, # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_efficientnet_b0_model()

        # --- Stage 1: freeze backbone, train classifier head ---
        print("\n--- Stage 1: Training classifier head ---")

        # --- 1.1. freeze feature extractor layers, except last block ---
        unfreeze_last_block_and_head(model)

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR_STAGE1,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1+1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_{PREFIX}_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model ---")

        # --- 2.1. unfreeze last two blocks + head ---
        unfreeze_last_two_blocks_and_head(model)

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR_STAGE2,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None

        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_{PREFIX}_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)   # restore best val-F1 weights
            print(f"Restored best Stage 2 weights for fold {fold} (F1={best_f1:.4f})")

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"{PREFIX}_fold{fold}.pth")


========== Fold 0 ==========

--- Stage 1: Training classifier head ---

Epoch 1/8


    t_loss=2.4618 | F1(macro)=0.3049 | Acc=0.3060


Confusion matrix:
 [[ 9  5 24  3]
 [18  2 12  0]
 [11  3 15  1]
 [ 5  0  7  2]]
Train  loss=2.4618 acc=0.3060 f1=0.3049 | Val loss=2.9860 acc=0.2393 f1=0.2126
  🔥 New best F1: 0.2126 – model saved.

Epoch 2/8


    t_loss=1.6719 | F1(macro)=0.3945 | Acc=0.3966


Confusion matrix:
 [[12 10 10  9]
 [11  4 10  7]
 [ 8  8  4 10]
 [ 5  2  2  5]]
Train  loss=1.6719 acc=0.3966 f1=0.3945 | Val loss=2.5206 acc=0.2137 f1=0.2049

Epoch 3/8


...

Confusion matrix:
 [[10  8 18  5]
 [ 4 13 13  2]
 [ 6  5 15  4]
 [ 1  1  7  4]]
Train  loss=1.1238 acc=0.5677 f1=0.5630 | Val loss=1.7363 acc=0.3621 f1=0.3526

Epoch 4/5


    t_loss=1.1354 | F1(macro)=0.5533 | Acc=0.5570


Confusion matrix:
 [[ 9  9 17  6]
 [ 2 14 12  4]
 [ 5  5 16  4]
 [ 4  0  5  4]]
Train  loss=1.1354 acc=0.5570 f1=0.5533 | Val loss=1.7387 acc=0.3707 f1=0.3550

Epoch 5/5


    t_loss=1.0719 | F1(macro)=0.5310 | Acc=0.5462


Confusion matrix:
 [[ 7  9 19  6]
 [ 5 11  9  7]
 [ 5  6 12  7]
 [ 2  1  3  7]]
Train  loss=1.0719 acc=0.5462 f1=0.5310 | Val loss=1.7888 acc=0.3190 f1=0.3212
Restored best Stage 2 weights for fold 4 (F1=0.3785)


# tf_efficientnetv2_s.in21k

In [6]:
def create_model_tf_efficientnetv2_s(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 10
    EPOCHS_STAGE2 = 15

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # False to disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_tf_efficientnetv2_s()

        # --- Stage 1: freeze backbone, train classifier head ---
        print("\n--- Stage 1: Training classifier head ---")

        # --- 1.1. freeze feature extractor layers ---
        for param in model.parameters():
            param.requires_grad = False

        # 2) unfreeze classifier head (EffNetV2 uses .classifier)
        for param in model.classifier.parameters():
            param.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1+1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model ---")

        # --- 2.1. unfreeze entire model ---
        for param in model.parameters():
            param.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None

        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)   # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")

# convnext_tiny

In [7]:
def create_model_convnext(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4  # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 12

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_convnext()

        # --- Stage 1: freeze backbone, train classifier HEAD (ConvNeXt) ---
        print("\n--- Stage 1: Training classifier head (ConvNeXt-Tiny) ---")

        # --- 1.1. freeze feature extractor layers ---
        for p in model.parameters():
            p.requires_grad = False

        # --- 1.1. unfreeze only the classifier head (ConvNeXt uses .head) ---
        for p in model.head.parameters():
            p.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model (ConvNeXt-Tiny) ---")

        # --- 2.1. unfreeze entire model ---
        for p in model.parameters():
            p.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)  # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)  # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"convnext_tiny_fold{fold}.pth")

# Model Inference with 5-Fold Ensembling

In [8]:
prefix_filename = "effv2_s" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S else "convnext_tiny" if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY else "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1"

# test_dataset = HistologyDataset(
#     df=test_df,
#     image_size=IMAGE_SIZE,
#     is_train=False,   # returns (img, sample_index)
#     use_mask_crop=True
# )
#
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=N_WORKERS,
#     pin_memory=cuda_is_available
# )
#
# all_fold_probs = []   # list of arrays [N, num_classes]
# all_sample_indices = None
#
# for fold in range(N_FOLDS):
#     print(f"Inference with fold {fold} model")
#
#     # recreate model and load weights
#     if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
#         model = create_model_tf_efficientnetv2_s(pretrained=False)
#     elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
#         model = create_model_convnext(pretrained=False)
#     else:
#         model = create_efficientnet_b0_model(pretrained=False)
#     state = torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device)
#     model.load_state_dict(state)
#     model.eval()
#
#     fold_probs = []
#     sample_indices_list = []
#
#     with torch.no_grad():
#         for imgs, sample_indices in test_loader:
#             imgs = imgs.to(device, non_blocking=True)
#
#             logits = model(imgs)               # [B, num_classes]
#             probs = softmax(logits, dim=1)     # [B, num_classes]
#             fold_probs.append(probs.cpu().numpy())
#
#             # collect sample indices only once
#             if all_sample_indices is None:
#                 sample_indices_list.extend(sample_indices)
#
#     fold_probs = np.concatenate(fold_probs, axis=0)  # [N, num_classes]
#     all_fold_probs.append(fold_probs)
#
#     if all_sample_indices is None:
#         all_sample_indices = sample_indices_list
#
# # average probabilities across folds
# mean_probs = np.mean(all_fold_probs, axis=0)   # [N, num_classes]
# pred_indices = mean_probs.argmax(axis=1)
#
# pred_labels = [idx2label[int(i)] for i in pred_indices]
# sample_index_with_ext = [
#     f"{si}.png" if not si.endswith(".png") else si
#     for si in all_sample_indices
# ]
#
# submission_df = pd.DataFrame({
#     "sample_index": sample_index_with_ext,
#     "label": pred_labels
# })
#
# submission_df.to_csv(f"submission_5fold_no_tta_{prefix_filename}.csv", index=False)
# print("Saved submission_5fold_no_tta.csv")
# print(submission_df.head())


In [9]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
all_fold_probs = []
all_sample_indices = None

test_dataset = HistologyDataset(
    df=test_df,
    image_size=IMAGE_SIZE,
    is_train=False,   # deterministic, returns (img, sample_index)
    use_mask_crop=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,               # per-image TTA
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

val_f1_per_fold = np.array(
    [best_f1_per_fold[fold] for fold in range(N_FOLDS)],
    dtype=np.float32
)

# Normalize to get weights that sum to 1
fold_weights = val_f1_per_fold / val_f1_per_fold.sum()
print("Fold weights:", fold_weights)

# -----------------------------
# 2) Accumulate weighted probs
# -----------------------------
all_probs = None
all_sample_indices = None

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model (weight={fold_weights[fold]:.3f})")

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    else:
        model = create_efficientnet_b0_model(pretrained=False)

    state_dict = torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for img_tensor, sample_idx in test_loader:
            # img_tensor: [1, 4, H, W]  (RGB+mask)
            img_tensor = img_tensor.squeeze(0).to(device)  # [4, H, W]

            # -------- TTA: apply multiple augmented views [4xHxW] --------
            tta_tensors = apply_tta(img_tensor)

            # accumulate probability predictions
            probs_sum = 0
            for aug_img in tta_tensors:
                aug_img = aug_img.unsqueeze(0).to(device)  # [1, 4, H, W]
                logits = model(aug_img)
                probs = softmax(logits, dim=1)  # [1, N_CLASSES]
                probs_sum += probs[0].cpu().numpy()

            # average across TTA views
            avg_probs = probs_sum / len(tta_tensors)  # [N_CLASSES]
            fold_probs.append(avg_probs)

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.append(sample_idx[0])

    fold_probs = np.vstack(fold_probs)  # [N_test, N_CLASSES]

    # initialize global probs
    if all_probs is None:
        all_probs = np.zeros_like(fold_probs, dtype=np.float32)

     # weighted accumulation
    all_probs += fold_weights[fold] * fold_probs

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# -----------------------------
# 3) Final predictions
# -----------------------------
pred_indices = all_probs.argmax(axis=1)
pred_labels = [idx2label[int(i)] for i in pred_indices]

sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})
submission_df.to_csv(f"submission_5fold_tta_{prefix_filename}.csv", index=False)

print(f"Saved submission_5fold_tta_{prefix_filename}.csv")

Fold weights: [0.18471073 0.15020123 0.21564856 0.20753433 0.2419052 ]
Inference with fold 0 model (weight=0.185)
Inference with fold 1 model (weight=0.150)
Inference with fold 2 model (weight=0.216)
Inference with fold 3 model (weight=0.208)
Inference with fold 4 model (weight=0.242)
Saved submission_5fold_tta_effb0.csv


In [10]:
def predict_loader_with_tta(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:  # note: here we have labels, not sample_index
            imgs = imgs.squeeze(0).to(device)  # if batch_size=1
            tta_imgs = apply_tta(imgs)         # same apply_tta as for test

            probs_sum = 0
            for aug in tta_imgs:
                aug = aug.unsqueeze(0).to(device)
                logits = model(aug)
                probs = softmax(logits, dim=1)
                probs_sum += probs[0].cpu().numpy()

            avg_probs = probs_sum / len(tta_imgs)
            all_probs.append(avg_probs)
            all_targets.append(labels.item())

    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)
    pred_indices = all_probs.argmax(axis=1)

    macro_f1 = f1_score(all_targets, pred_indices, average="macro")
    return macro_f1

fold_f1s = []

for fold in range(N_FOLDS):
    print(f"OOF eval for fold {fold}")

    # build val_df_split for that fold
    val_df_split = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_dataset = HistologyDataset(
        df=val_df_split,
        image_size=IMAGE_SIZE,
        is_train=False,   # Disable augmentations
        use_mask_crop=True
    )
    val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=N_WORKERS, pin_memory=cuda_is_available)

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    else:
        model = create_efficientnet_b0_model(pretrained=False)
    model.load_state_dict(torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device))

    f1 = predict_loader_with_tta(model, val_loader, device)
    fold_f1s.append(f1)
    print("Fold F1 (OOF, with TTA):", f1)

print("Mean OOF F1:", np.mean(fold_f1s))


OOF eval for fold 0
Fold F1 (OOF, with TTA): 0.3398613469156763
OOF eval for fold 1
Fold F1 (OOF, with TTA): 0.21390253890253888
OOF eval for fold 2
Fold F1 (OOF, with TTA): 0.2526581605528974
OOF eval for fold 3
Fold F1 (OOF, with TTA): 0.2711349924585219
OOF eval for fold 4
Fold F1 (OOF, with TTA): 0.4104180677317791
Mean OOF F1: 0.29759502131228277
